In [1]:
from pathlib import Path
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

SEED = 42
pd.set_option("display.max_colwidth", 120) 

In [2]:
PROYECTO_DIR = Path(".").resolve()
TRAIN_PATH = PROYECTO_DIR / "train.csv"
EVAL_PATH = PROYECTO_DIR / "eval.csv"
OUTPUT_DIR = PROYECTO_DIR / "outputs_parte1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROYECTO_DIR:", PROYECTO_DIR)

train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)

print("train_df shape:", train_df.shape)
print("eval_df shape:", eval_df.shape)

display(train_df.head())
display(eval_df.head())

PROYECTO_DIR: C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto
train_df shape: (31403, 2)
eval_df shape: (3490, 2)


,text,decade
0,"\nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de poreft.Proreg.118,3, $.9.M.-70 \npag.4.1. 3 Ste ph.Gratian. difcept.2...",164
1,"gone. Sus amigos , sus clientes, todo \ncuanto le rodea le prueban que es hom- \nbre de mucha importancia. Si ve que...",182
2,"Prefosen quemanera,e per qualesfolpechas deuan feratormentados, e \nante quien,e quie preguntasles deuan hazer mient...",157
3,"Caistro el M a y o r a i .] Del apellido de Cañroíe \nvalió Don Luis enefta. Metáfora, para el no...",163
4,"\nlos que panden macho ; y \notros en la fu ña abundan¬ \ncia , íiempre viuen como \nmendigos. La La...",166


,id,text
0,0,"P. Si en efta convocación trato folamente de comunicarles los artículos arregla- \ndos, y firmados, o de..."
1,1,"«Muy santo Padre : Ayer escribiá don Juan Man- \ny rique que dijese á vuestra Santidad, ó le escribie- \n»se, en cuá..."
2,2,"Recibo, otorgado por Diego Gracián a favor de Jerónimo Zurita, de un libro de Casiodoro, titulado \nDe Amicitia et D..."
3,3,"6. Los Samaritanos no admitían por \nEfcrituras Canónicas fino los cinco Libros \nde Moyfes, tfloes, Gen..."
4,4,ü«mi yofacaremisoiiejasfilasmaí \nI nosodlosrt no confeti tire q mas fe las co* \n( mauXomeímo oíjeeiielfal...


In [4]:
required_train_cols = {"text", "decade"}
required_eval_cols = {"id", "text"}

if not required_train_cols.issubset(train_df.columns):
    raise ValueError(f"train.csv debe contener columnas {required_train_cols}")
if not required_eval_cols.issubset(eval_df.columns):
    raise ValueError(f"eval.csv debe contener columnas {required_eval_cols}")

print("Columnas train:", list(train_df.columns))
print("Columnas eval:", list(eval_df.columns))

print("Nulos en train:")
display(train_df.isnull().sum().to_frame("nulls"))
print("Nulos en eval:")
display(eval_df.isnull().sum().to_frame("nulls"))

dup_rows_train = train_df.duplicated().sum()
dup_rows_eval = eval_df.duplicated().sum()
dup_text_train = train_df["text"].duplicated().sum()

print(f"Filas duplicadas en train: {dup_rows_train}")
print(f"Filas duplicadas en eval: {dup_rows_eval}")
print(f"Textos duplicados en train: {dup_text_train}")

conflictos_textos = (
    train_df.groupby("text")["decade"].nunique().reset_index(name="n_labels").query("n_labels > 1")
)

print("Textos iguales con etiquetas distintas:", len(conflictos_textos))

print("\nNumero de clases:", train_df["decade"].nunique())
print("Rango de decadas:", train_df["decade"].min(), "-", train_df["decade"].max())

Columnas train: ['text', 'decade']
Columnas eval: ['id', 'text']
Nulos en train:


,nulls
text,0
decade,0


Nulos en eval:


,nulls
id,0
text,0


Filas duplicadas en train: 34
Filas duplicadas en eval: 0
Textos duplicados en train: 51
Textos iguales con etiquetas distintas: 12

Numero de clases: 39
Rango de decadas: 150 - 188


Se realizó una revisión básica del conjunto de entrenamiento y evaluación:
- verificación de columnas requeridas,
- inspección de valores nulos,
- revisión de duplicados exactos y textos repetidos,
- revisión del número de clases y su rango.

En esta versión no se eliminaron duplicados automáticamente, porque existen textos repetidos y algunos casos con etiquetas distintas; removerlos sin criterio adicional podría alterar el problema original. La decisión metodológica fue conservar el dataset original y documentar esta condición.

In [5]:
x_train = train_df[["text"]].fillna("")
y_train = train_df["decade"].astype(int)

x_eval = eval_df[["text"]].fillna("")
eval_ids = eval_df["id"]

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_eval:", x_eval.shape)

x_train: (31403, 1)
y_train: (31403,)
x_eval: (3490, 1)


In [6]:
def build_preprocessor(
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 5),
    word_min_df=3,
    char_min_df=3,
    word_max_features=120_000,
    char_max_features=180_000,
):
    return ColumnTransformer(
        transformers=[
            (
                "word_tfidf",
                TfidfVectorizer(
                    strip_accents="unicode",
                    lowercase=True,
                    ngram_range=word_ngram_range,
                    min_df=word_min_df,
                    max_df=0.95,
                    sublinear_tf=True,
                    max_features=word_max_features,
                ),
                "text",
            ),
            (
                "char_tfidf",
                TfidfVectorizer(
                    strip_accents="unicode",
                    lowercase=True,
                    analyzer="char_wb",
                    ngram_range=char_ngram_range,
                    min_df=char_min_df,
                    sublinear_tf=True,
                    max_features=char_max_features,
                ),
                "text",
            ),
        ],
        remainder="drop",
    )

def make_pipeline(model, **prep_kwargs):
    return Pipeline([
        ("features", build_preprocessor(**prep_kwargs)),
        ("model", model),
    ])
    
def build_models_submission_1():
    return {
        "LinearSVC_C1.5": make_pipeline(LinearSVC(C=1.5, random_state=SEED)),
        "LinearSVC_C2.5": make_pipeline(LinearSVC(C=2.5, random_state=SEED)),
        "LogReg_C3": make_pipeline(LogisticRegression(C=3.0, max_iter=3000, random_state=SEED)),
        "ComplementNB_a025": make_pipeline(ComplementNB(alpha=0.25)),
    }
    
def build_models_submission_2():
    return {
        "LogReg_C1" : make_pipeline(LogisticRegression(C=1.0, max_iter=3000, random_state=SEED)),
        "LogReg_C3" : make_pipeline(LogisticRegression(C=3.0, max_iter=3000, random_state=SEED)),
        "LogReg_C5" : make_pipeline(LogisticRegression(C=5.0, max_iter=3000, random_state=SEED)),
        "LogReg_C7" : make_pipeline(LogisticRegression(C=7.0, max_iter=3000, random_state=SEED)),
    }

In [7]:
def run_experiments(models_dict, experiment_name):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []
    
    print(f"Ejecutando experimentos: {experiment_name}")
    
    for name ,model in models_dict.items():
        t0 = time.time()
        
        cv_results = cross_validate(
            model,
            x_train,
            y_train,
            cv=cv,
            scoring="accuracy",
            n_jobs=1,
            return_train_score=False,
        )
        
        elapsed = time.time() - t0
        
        rows.append({
            "experiment": experiment_name,
            "model": name,
            "cv_accuracy_mean": float(cv_results["test_score"].mean()),
            "cv_accuracy_std": float(cv_results["test_score"].std()),
            "elapsed_sec": round(elapsed, 1),
        })
        
        print(
            f"{name}: "
            f"{rows[-1]["cv_accuracy_mean"]:.5f} "
            f"+/- {rows[-1]["cv_accuracy_std"]:.5f}) "
            f"({elapsed:.1f}s)"
        )
    
    results = (
        pd.DataFrame(rows)
        .sort_values("cv_accuracy_mean", ascending=False)
        .reset_index(drop=True)
    )
    
    out_path = OUTPUT_DIR / f"cv_resultados_{experiment_name}.csv"
    results.to_csv(out_path, index=False)
    
    print(f"Resultados guardados en: {out_path}")
    display(results)
    
    return results

In [8]:
models_s1 = build_models_submission_1()
cv_results_s1 = run_experiments(models_s1, "submission_01_baseline")

Ejecutando experimentos: submission_01_baseline
LinearSVC_C1.5: 0.25322 +/- 0.00276) (394.6s)
LinearSVC_C2.5: 0.25192 +/- 0.00261) (476.2s)
LogReg_C3: 0.25998 +/- 0.00294) (1512.7s)
ComplementNB_a025: 0.22663 +/- 0.00493) (180.5s)
Resultados guardados en: C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto\outputs_parte1\cv_resultados_submission_01_baseline.csv


,experiment,model,cv_accuracy_mean,cv_accuracy_std,elapsed_sec
0,submission_01_baseline,LogReg_C3,0.259975,0.002940,1512.7
1,submission_01_baseline,LinearSVC_C1.5,0.253224,0.002757,394.6
2,submission_01_baseline,LinearSVC_C2.5,0.251919,0.002613,476.2
3,submission_01_baseline,ComplementNB_a025,0.226635,0.004927,180.5


In [9]:
best_model_name_s1 = cv_results_s1.loc[0, "model"]
best_model_s1 =models_s1[best_model_name_s1]

best_model_s1.fit(x_train, y_train)

model_path_s1 = OUTPUT_DIR / f"modelo_submission_01_baseline.joblib"
joblib.dump(best_model_s1, model_path_s1)

preds_s1 = best_model_s1.predict(x_eval)
submission_s1 = pd.DataFrame({
    "id": eval_ids,
    "answer": preds_s1.astype(int),
})

submission_path_s1 = OUTPUT_DIR / "submission_01_baseline.csv"
submission_s1.to_csv(submission_path_s1, index=False)

print("Modelo Submission 1:", best_model_name_s1)
print("Modelo guardado en :", model_path_s1)
print("Submission guardada:", submission_path_s1)
print("Columnas:", list(submission_s1.columns))
print("Filas:", len(submission_s1))

display(submission_s1.head())

Modelo Submission 1: LogReg_C3
Modelo guardado en : C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto\outputs_parte1\modelo_submission_01_baseline.joblib
Submission guardada: C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto\outputs_parte1\submission_01_baseline.csv
Columnas: ['id', 'answer']
Filas: 3490


,id,answer
0,0,173
1,1,187
2,2,150
3,3,172
4,4,153


In [10]:
models_s2 = build_models_submission_2()
cv_results_s2 = run_experiments(models_s2, "submission_02_logreg_tuning")

Ejecutando experimentos: submission_02_logreg_tuning
LogReg_C1: 0.25498 +/- 0.00433) (990.2s)
LogReg_C3: 0.25998 +/- 0.00294) (1324.2s)
LogReg_C5: 0.25918 +/- 0.00311) (1951.7s)
LogReg_C7: 0.25940 +/- 0.00304) (1504.9s)
Resultados guardados en: C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto\outputs_parte1\cv_resultados_submission_02_logreg_tuning.csv


,experiment,model,cv_accuracy_mean,cv_accuracy_std,elapsed_sec
0,submission_02_logreg_tuning,LogReg_C3,0.259975,0.002940,1324.2
1,submission_02_logreg_tuning,LogReg_C7,0.259402,0.003039,1504.9
2,submission_02_logreg_tuning,LogReg_C5,0.259179,0.003114,1951.7
3,submission_02_logreg_tuning,LogReg_C1,0.254976,0.004330,990.2


In [11]:
best_model_name_s2 = cv_results_s2.loc[0, "model"]
best_model_s2 = models_s2[best_model_name_s2]

best_model_s2.fit(x_train, y_train)

model_path_s2 = OUTPUT_DIR / f"modelo_submission_02_logreg_tuning.joblib"
joblib.dump(best_model_s2, model_path_s2)

preds_s2 = best_model_s2.predict(x_eval)
submission_s2 = pd.DataFrame({
    "id": eval_ids,
    "answer": preds_s2.astype(int),
})

submission_path_s2 = OUTPUT_DIR / "submission_02_logreg_tuning.csv"
submission_s2.to_csv(submission_path_s2, index=False)

print("Modelo Submission 2:", best_model_name_s2)
print("Modelo guardado en :", model_path_s2)
print("Submission guardada:", submission_path_s2)
print("Columnas:", list(submission_s2.columns))
print("Filas:", len(submission_s2))

display(submission_s2.head())

Modelo Submission 2: LogReg_C3
Modelo guardado en : C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto\outputs_parte1\modelo_submission_02_logreg_tuning.joblib
Submission guardada: C:\ANDES\BI\MaterialDeClase-ISIS-2611\Proyecto\outputs_parte1\submission_02_logreg_tuning.csv
Columnas: ['id', 'answer']
Filas: 3490


,id,answer
0,0,173
1,1,187
2,2,150
3,3,172
4,4,153
